# Pipeline audit metrics

This notebook loads the archived result artifacts used by `README.md`. It deliberately does not rerun model inference. Every displayed value is measured unless a note explicitly says otherwise. Run it from the repository root.

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'notebooks/archive').exists():
    ROOT = Path('../..').resolve()
ARCHIVE = ROOT / 'notebooks/archive'
CURRENT = ARCHIVE / 'results/current'
LVLM = ARCHIVE / 'results/lvlm'
assert CURRENT.exists() and LVLM.exists()

## Action-generation prompt lineage (90 human-labeled audit-v11 clips)

In [ ]:
versions = {
    'base': 'qwen3.5_audit_v11_seg_extract_spec_eval.spec_f1_overall_aggregate.csv',
    'CVS context v1': 'qwen3.5_audit_v11_seg_extract_spec_eval_with_cvs_context.spec_f1_overall_aggregate.csv',
    'CVS context v2': 'qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v2__baseline.spec_f1_overall_aggregate.csv',
    'CVS context v3': 'qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3.spec_f1_overall_aggregate.csv',
    'CVS context v4': 'qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v4.spec_f1_overall_aggregate.csv',
}
rows = []
for version, filename in versions.items():
    frame = pd.read_csv(CURRENT / filename)
    score_col = 'aggregate mean F1' if 'aggregate mean F1' in frame else 'f1_mean_avg'
    count_col = 'rows' if 'rows' in frame else 'n_rows'
    for _, row in frame.iterrows():
        rows.append({'version': version, 'method': row['method'], 'spec_f1': row[score_col], 'summary_rows': row[count_col], 'source': filename})
lineage = pd.DataFrame(rows)
lineage.pivot(index='version', columns='method', values='spec_f1').round(3)

In [ ]:
actor_rows = []
for version, filename in versions.items():
    actor_filename = filename.replace('spec_f1_overall_aggregate', 'spec_f1_actor_aggregate_table')
    frame = pd.read_csv(CURRENT / actor_filename)
    if 'method' in frame.columns:
        method_name = 'structured_prediction' if version == 'base' else 'structured_prediction_cvs_context'
        frame = frame[frame['method'] == method_name]
        for _, row in frame.iterrows():
            actor_rows.append({'version': version, 'actor': row['actor'], 'structured_actor_f1': row['f1_mean_avg']})
    else:
        for _, row in frame.iterrows():
            actor_rows.append({'version': version, 'actor': row['actor'], 'structured_actor_f1': row['structured_prediction_cvs_context']})
pd.DataFrame(actor_rows).pivot(index='version', columns='actor', values='structured_actor_f1').round(3)

v3.1 and v3.2 are post-processing variants of v3. The next cells recompute their structured-prediction aggregate F1 from the archived evaluation rows and count the changes recorded by their override logs.

In [ ]:
postprocess_scores = []
for version in ('v3_1', 'v3_2'):
    data = json.loads((CURRENT / f'{version}_spec_eval_results.json').read_text())
    rows = data['structured_prediction_cvs_context']
    right_rows = [row for row in rows if row['actor'] == 'right']
    postprocess_scores.append({
        'version': version.replace('_', '.'),
        'structured_spec_f1': sum(row['f1_mean'] for row in rows) / len(rows),
        'structured_right_actor_f1': sum(row['f1_mean'] for row in right_rows) / len(right_rows),
        'summary_rows': len(rows),
    })
pd.DataFrame(postprocess_scores).round(6)

In [ ]:
override_rows = []
for version in ('v3_1', 'v3_2'):
    data = json.loads((CURRENT / f'{version}_override_summary.json').read_text())
    override_rows.append({
        'version': version.replace('_', '.'),
        'logged_rows': len(data),
        'examples': len({row['example_id'] for row in data}),
        'overrides_applied': sum(bool(row['override_applied']) for row in data),
    })
pd.DataFrame(override_rows)

In [ ]:
full_override_rows = []
for split in ('train_full', 'test_full'):
    data = json.loads((CURRENT / f'v3_1_{split}_override_summary.json').read_text())
    full_override_rows.append({
        'split': split,
        'logged_rows': len(data),
        'examples': len({row['example_id'] for row in data}),
        'rubric_predictions_present': sum(bool(row.get('rubric_clip_tool')) for row in data),
        'overrides_applied': sum(bool(row['override_applied']) for row in data),
    })
pd.DataFrame(full_override_rows)

## Selected v3.1 held-out results

In [ ]:
scores = pd.read_csv(CURRENT / 'validation_test_final_scores_seed20260610.csv')
selected = scores[scores.method_id == 'structured_llm_extract_tool_fix'][
    ['split', 'left_exact', 'right_medium', 'camera_coarse', 'final_score']
].copy()
selected.round(3)

In [ ]:
corr = pd.read_csv(CURRENT / 'recomputed_six_system_human_synthetic_final_score_correlation_test20.csv')
corr[['n_systems', 'n_test_videos_in_seed_split', 'pearson_r', 'pearson_bootstrap_ci_low', 'pearson_bootstrap_ci_high', 'spearman_r', 'spearman_bootstrap_ci_low', 'spearman_bootstrap_ci_high']].round(3)

The direct scores and the correlation answer different questions: direct scores estimate agreement with human action labels; the six-point correlation estimates whether synthetic GT ranks six downstream systems similarly.

## Additional right-tool checker

In [ ]:
right_tool_nb = json.loads((ARCHIVE / 'experimental_action_pipeline/qwen3.5_audit_v11_right_tool_rubric_clip_eval_1fps.ipynb').read_text())
all_output_text = '\n'.join(
    ''.join(output.get('text', [])) + ''.join(output.get('data', {}).get('text/plain', []))
    for cell in right_tool_nb['cells'] for output in cell.get('outputs', [])
)
assert '0.568182' in all_output_text and '0.590909' in all_output_text
pd.DataFrame([
    {'method': 'v3 structured clip tool', 'n': 88, 'accuracy': 0.590909},
    {'method': 'rubric + 10 shape examples', 'n': 88, 'accuracy': 0.568182},
]).round(3)

## Earlier `lvlm_cvs_reasoning` ablations

In [ ]:
lvlm_runs = {
    'single / direct / no organs': 'vanilla_gpt-4.1-mini_train_dev5_frames_pred.metrics.json',
    'sequential / direct / no organs': 'sequential_inductive_gpt-4.1-mini_train_dev5_frames_pred.metrics.json',
    'single / direct / organs': 'vanilla_organs_gpt-4.1-mini_train_dev5_frames_pred.metrics.json',
    'sequential / direct / organs': 'sequential_inductive_organs_gpt-4.1-mini_train_dev5_frames_pred.metrics.json',
    'single / observe-then-decide / no organs': 'vanilla_gpt-4.1-mini_train_dev5_frames_obs_then_decide.metrics.json',
    'sequential / observe-then-decide / no organs': 'sequential_inductive_gpt-4.1-mini_train_dev5_frames_obs_then_decide.metrics.json',
}
lvlm_rows = []
for label, filename in lvlm_runs.items():
    row = json.loads((LVLM / filename).read_text())
    lvlm_rows.append({'configuration': label, **row, 'source': filename})
lvlm = pd.DataFrame(lvlm_rows)
lvlm[['configuration', 'num_frames', 'AP_c1', 'AP_c2', 'AP_c3', 'mAP_criteria']].round(3)

In [ ]:
organs = pd.read_csv(LVLM / 'organ_sweep_summary.csv')
organs[['organ_variant', 'num_frames', 'AP_c1', 'AP_c2', 'AP_c3', 'mAP_criteria']].sort_values('mAP_criteria', ascending=False).round(3)

These earlier results are a separate CVS-achievement task, not action-segmentation accuracy. They influenced the design choice to include temporal/CVS context, but their mAP values must not be presented as v3.1 action-label performance. The experiments use only 90 frames from five development videos, so qualitative claims about temporal prompting or organ overlays are exploratory, not confirmed on an independent test set.